# Lab 2 | ADF + sai phân

**Mục bình chọn:** ADF + sai phân — cả 2 dataset

**Hạn bàn giao:** 23/10/2026, 20:00 — bàn giao phép xử lý và d/D đề xuất.

> Bạn kiểm tra chuỗi có bằng chứng dừng hay chưa, sau đó thử sai phân và kiểm định lại. Người làm ACF/PACF cần biết chính xác bạn đã xử lý chuỗi như thế nào để tiếp tục, không chỉ nhận một con số p-value.

**Đối chiếu đề:** Bài cơ bản: Bài 2–3 trên Dataset 1. Nâng cao: ADF chuỗi gốc, sai phân thường và mùa vụ trên Dataset 2.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần có:** dữ liệu train của cả hai dataset và quy ước chia 80/20.
**Có thể làm ngay**, song song với EDA; không phải chờ phần EDA viết xong nhận xét.

## Cụ thể cần làm gì?

1. Thực hiện ADF trên chuỗi train gốc của cả hai dataset; ghi ADF statistic, p-value,
   mức ý nghĩa, cách chọn lag và kết luận bằng lời.
2. Dataset 1: sai phân bậc 1 theo đề, xử lý giá trị thiếu do sai phân, vẽ chuỗi sau xử lý và ADF lại.
3. Dataset 2: kiểm tra sai phân thường và sai phân mùa vụ với chu kỳ 12; lưu kết quả trước/sau xử lý.
   d=1, D=1 trong đề là gợi ý cần kiểm tra, không phải kết luận điền sẵn.
4. Đề xuất d cho Dataset 1, d/D cho Dataset 2 kèm bằng chứng; nếu còn vấn đề thì ghi rõ,
   không ép kết luận “đã dừng”.
5. Trả lời câu hỏi: H₀ của ADF là gì; p-value nói gì; chuỗi trước/sau sai phân ra sao;
   vì sao đề xuất bậc sai phân đó.

## Cần bàn giao những gì?

**Gửi cho phần ACF/PACF và mô hình:**
- Bảng ADF trước/sau xử lý cho hai dataset, đồ thị sau sai phân và nhận xét.
- Phép biến đổi theo đúng thứ tự, bậc sai phân, chu kỳ mùa vụ và cách bỏ NaN.
- Code tái tạo chuỗi đã xử lý cùng thông tin index; không chỉ gửi biến đang có trong kernel.
- Nêu rõ: chuỗi đã xử lý dùng cho ACF/PACF; khi fit ARIMA/SARIMA trên train gốc,
  mô hình sử dụng d/D tương ứng, tránh sai phân lặp hai lần.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [ ] Có kết quả cho cả Dataset 1 và Dataset 2.
- [ ] Có kiểm định lại sau xử lý, giải thích d/D và trả lời câu hỏi theo đề.
- [ ] Phần ACF/PACF tái tạo được đúng chuỗi bạn đã dùng.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. ADF chuỗi gốc của hai dataset

Kiểm định ADF chuỗi gốc trên train1 và train2.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent          # chạy từ thư mục contributions/
sys.path.insert(0, str(ROOT))
from src.lab2_data import load_datasets, chronological_split, SEASON_LENGTH

_, _, y1, y2 = load_datasets()
train1, test1 = chronological_split(y1)   # test chỉ để kiểm tra kích thước, không dùng chọn d/D
train2, test2 = chronological_split(y2)
print(f'Dataset 1 (a10)       : train={len(train1)}, test={len(test1)}, {train1.index[0]:%Y-%m} -> {train1.index[-1]:%Y-%m}')
print(f'Dataset 2 (Passengers): train={len(train2)}, test={len(test2)}, {train2.index[0]:%Y-%m} -> {train2.index[-1]:%Y-%m}')

ALPHA = 0.05


def adf_row(series, name, regression='c'):
    """ADF với autolag='AIC' (maxlag mặc định của statsmodels), trả về một dòng bảng kết quả."""
    s = series.dropna()
    stat, p, usedlag, nobs, crit, _ = adfuller(s, regression=regression, autolag='AIC', result_object=False)
    return {
        'Chuỗi': name, 'n': int(nobs) + usedlag + 1, 'regression': regression,
        'ADF stat': round(stat, 4), 'p-value': round(p, 4), 'lag (AIC)': usedlag,
        'crit 1%': round(crit['1%'], 3), 'crit 5%': round(crit['5%'], 3),
        'Kết luận (α=0.05)': 'Dừng (bác bỏ H0)' if p < ALPHA else 'Chưa dừng (không bác bỏ H0)',
    }


adf_orig = pd.DataFrame([
    adf_row(train1, 'D1 a10 - gốc'),
    adf_row(train2, 'D2 Passengers - gốc'),
])
adf_orig

**Nhận xét (ADF chuỗi gốc, chỉ dùng train):**

- **H₀ của ADF:** chuỗi có nghiệm đơn vị (không dừng). **H₁:** chuỗi dừng. **p-value** là xác suất thu được thống kê ADF cực đoan như đã quan sát nếu H₀ đúng: p < 0.05 thì bác bỏ H₀ (kết luận dừng), p ≥ 0.05 thì không đủ bằng chứng để bác bỏ. Dùng `regression='c'` (có hằng số), `autolag='AIC'` chọn lag tối đa tự động; mức ý nghĩa α = 0.05.
- **Dataset 1 (a10):** ADF = 2.23, p = 0.9989, lag AIC = 14 → **không bác bỏ H₀**, chuỗi chưa dừng. Thống kê dương, lớn hơn rất nhiều so với giá trị tới hạn 5% (−2.88): chuỗi tăng mạnh và có biên độ mùa vụ tăng dần.
- **Dataset 2 (Passengers):** ADF = −0.36, p = 0.9171, lag AIC = 13 → **không bác bỏ H₀**, chuỗi chưa dừng (xu hướng tăng + mùa vụ nhân tính).
- **Hạn chế:** lag AIC chọn (14 và 13) **trùng đúng giá trị maxlag mặc định** của statsmodels, tức là AIC bị chặn trên bởi maxlag. Đã kiểm tra lại ở mục 2 với `maxlag=24`, BIC và `regression='ct'`: p-value vẫn rất lớn (≥ 0.47 với chuỗi gốc) nên kết luận "chưa dừng" không phụ thuộc cách chọn lag. "Không bác bỏ H₀" chỉ nghĩa là *không có đủ bằng chứng để coi là dừng*, không chứng minh có nghiệm đơn vị; ADF cũng có công suất thấp khi chuỗi có mùa vụ chu kỳ 12.

### 2. Sai phân, đồ thị và kiểm định lại

Sai phân, vẽ lại và kiểm định lại; đề xuất d/D cùng bằng chứng.

In [ ]:
def diff1(s):
    return s.diff(1).dropna()


def diff_season(s, m=SEASON_LENGTH):
    return s.diff(m).dropna()


def diff_both(s, m=SEASON_LENGTH):
    return s.diff(1).diff(m).dropna()      # (1-B)(1-B^12) y_t; thứ tự giao hoán nên không ảnh hưởng kết quả


# Dataset 1: sai phân bậc 1 theo đề; thêm sai phân mùa vụ để đối chiếu (a10 có mùa vụ rất rõ)
t1 = {
    'D1 d=1': diff1(train1),
    'D1 D=1 (m=12)': diff_season(train1),
    'D1 d=1, D=1': diff_both(train1),
}
# Dataset 2: sai phân thường, mùa vụ, và cả hai
t2 = {
    'D2 d=1': diff1(train2),
    'D2 D=1 (m=12)': diff_season(train2),
    'D2 d=1, D=1': diff_both(train2),
}
transformed = {**t1, **t2}

adf_diff = pd.DataFrame([adf_row(s, n) for n, s in transformed.items()])
display(adf_diff)

# Số điểm mất do sai phân (NaN bị loại bằng dropna, index giữ nguyên theo thời gian)
loss = pd.DataFrame({
    'n gốc': [len(train1)] * 3 + [len(train2)] * 3,
    'n sau xử lý': [len(s) for s in transformed.values()],
    'bắt đầu từ': [f'{s.index[0]:%Y-%m}' for s in transformed.values()],
    'std': [round(s.std(), 3) for s in transformed.values()],   # std tăng khi sai phân thêm => dấu hiệu sai phân thừa
}, index=transformed.keys())
display(loss)

# Độ nhạy của kết luận ADF (p-value) theo cách chọn lag và thành phần tất định
def adf_p(s, **kw):
    return round(adfuller(s.dropna(), result_object=False, **kw)[1], 3)


all_series = {'D1 gốc': train1, 'D2 gốc': train2, **transformed}
adf_sens = pd.DataFrame({
    'AIC (mặc định)': {n: adf_p(s, autolag='AIC') for n, s in all_series.items()},
    'BIC': {n: adf_p(s, autolag='BIC') for n, s in all_series.items()},
    'AIC, maxlag=24': {n: adf_p(s, autolag='AIC', maxlag=24) for n, s in all_series.items()},
    "AIC, regression='ct'": {n: adf_p(s, autolag='AIC', regression='ct') for n, s in all_series.items()},
})
display(adf_sens)

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
for col, (orig, group, title) in enumerate([(train1, t1, 'Dataset 1 (a10)'), (train2, t2, 'Dataset 2 (Passengers)')]):
    axes[0, col].plot(orig); axes[0, col].set_title(f'{title} - train gốc')
    for row, (name, s) in enumerate(group.items(), start=1):
        axes[row, col].plot(s); axes[row, col].axhline(0, color='grey', lw=0.8, ls='--')
        axes[row, col].set_title(name)
fig.tight_layout()
plt.show()

**Nhận xét (sai phân và kiểm định lại):**

- **Dataset 1:** sai phân bậc 1 (theo đề) làm ADF cải thiện (2.23 → −1.71) nhưng p = 0.428 → **vẫn chưa dừng**. Nguyên nhân là mùa vụ chu kỳ 12 còn nguyên (đồ thị sai phân vẫn dao động đều theo năm với biên độ tăng). Sai phân mùa vụ riêng lẻ cũng chưa đủ (p = 0.94) vì còn xu hướng; chỉ khi **d=1 và D=1** thì ADF = −8.79, p ≈ 0 → dừng.
- **Dataset 2:** sai phân thường (d=1) p = 0.106 → chưa dừng ở α = 0.05. Sai phân mùa vụ (D=1) cho p = 0.013 → dừng ở 5% nhưng **không** ở 1% (ADF −3.34 > −3.50), tức là còn biên. Kết hợp **d=1, D=1** cho ADF = −12.18, p ≈ 0, lag AIC = 0 → dừng rõ ràng.
- Mỗi lần sai phân mất dữ liệu đầu chuỗi (1 điểm cho d, 12 điểm cho D); các NaN được `dropna()` và index thời gian của phần còn lại được giữ nguyên (xem bảng "bắt đầu từ").
- **Kiểm tra sai phân thừa (cột `std`):** D2 giảm đều (26.8 → 15.8 → 9.9) nên d=1+D=1 không có dấu hiệu sai phân thừa. D1 lại tăng nhẹ từ 0.690 (chỉ D=1) lên 0.736 (d=1+D=1), gợi ý d=1 có thể hơi thừa, nhưng ADF của D=1 riêng lẻ lại không bác bỏ H₀ (p = 0.94) nên vẫn cần d=1 theo bằng chứng ADF; cần đối chiếu thêm ACF/PACF.
- **Độ nhạy (bảng `adf_sens`):** kết luận cho d=1+D=1 vững ở cả hai dataset (p ≈ 0 với mọi cách chọn lag, kể cả `regression='ct'`). Ngược lại D2 với D=1 riêng lẻ **không vững**: p = 0.013 (AIC), 0.007 (BIC) nhưng 0.050 khi thêm xu hướng (`ct`), tức đúng ngưỡng 5%; vì vậy không nên dựa vào D=1 riêng lẻ cho D2.
- **Hạn chế:** đồ thị cho thấy phương sai chưa ổn định hoàn toàn: D1 sau d=1+D=1 có biên độ dao động tăng rõ từ khoảng năm 2000, D2 dao động mạnh hơn ở 1952–1954 → có thể cần log/Box–Cox; chưa thực hiện vì đề không yêu cầu. ADF chỉ kiểm tra nghiệm đơn vị, không loại trừ mùa vụ tất định còn lại.

### 3. Kết luận d/D và bàn giao phép xử lý

Bàn giao phép xử lý chính xác cho phần ACF/PACF.

In [ ]:
# Kết luận d/D dựa trên bảng ADF ở trên (chỉ dùng train). Điền sau khi xem kết quả.
FINAL = {
    'D1': {'d': 1, 'D': 1, 'm': SEASON_LENGTH},
    'D2': {'d': 1, 'D': 1, 'm': SEASON_LENGTH},
}


def make_stationary(series, d, D, m=SEASON_LENGTH):
    """Tái tạo chuỗi đã xử lý: sai phân thường d lần rồi sai phân mùa vụ D lần (lag m), bỏ NaN đầu chuỗi."""
    out = series.copy()
    for _ in range(d):
        out = out.diff(1)
    for _ in range(D):
        out = out.diff(m)
    return out.dropna()


stat1 = make_stationary(train1, FINAL['D1']['d'], FINAL['D1']['D'])
stat2 = make_stationary(train2, FINAL['D2']['d'], FINAL['D2']['D'])

# Kiểm tra tái tạo khớp với phép biến đổi ở mục 2
assert stat1.equals(transformed['D1 d=1, D=1'])
assert stat2.equals(transformed['D2 d=1, D=1'])

summary = pd.DataFrame([
    {'Dataset': 'D1 a10', **FINAL['D1'], 'n': len(stat1), 'index': f'{stat1.index[0]:%Y-%m} -> {stat1.index[-1]:%Y-%m}',
     'ADF p-value': adf_row(stat1, 'D1')['p-value']},
    {'Dataset': 'D2 Passengers', **FINAL['D2'], 'n': len(stat2), 'index': f'{stat2.index[0]:%Y-%m} -> {stat2.index[-1]:%Y-%m}',
     'ADF p-value': adf_row(stat2, 'D2')['p-value']},
])
display(summary)

# Lưu chuỗi đã xử lý (có index thời gian) để phần ACF/PACF đọc lại; tables/ được Git bỏ qua
out_dir = ROOT / 'tables'
out_dir.mkdir(exist_ok=True)
stat1.rename('stationary_d1').to_csv(out_dir / 'stationary_dataset1_train.csv', index_label='date')
stat2.rename('stationary_d2').to_csv(out_dir / 'stationary_dataset2_train.csv', index_label='date')
pd.concat([adf_orig, adf_diff]).to_csv(out_dir / 'adf_results_02.csv', index=False)
print('Đã lưu vào', out_dir)

**Nhận xét (kết luận d/D và bàn giao):**

| Dataset | Đề xuất | Bằng chứng (train, ADF) |
|---|---|---|
| D1 a10 | **d=1, D=1, m=12** | gốc p=0.999; d=1 p=0.428; D=1 p=0.940; d=1+D=1 p≈0 |
| D2 Passengers | **d=1, D=1, m=12** | gốc p=0.917; d=1 p=0.106; D=1 p=0.013 (biên); d=1+D=1 p≈0 |

- Với **D1**, d=1 như đề gợi ý là **chưa đủ**: cần thêm D=1; chỉ riêng sai phân bậc 1 không cho chuỗi dừng, nên không ép kết luận "đã dừng". Lưu ý std tăng nhẹ (0.690 → 0.736) khi thêm d=1 vào D=1, nên d=1 là phần cần xem lại bằng ACF/PACF.
- Với **D2**, d=1, D=1 như đề gợi ý được ủng hộ bởi dữ liệu; D=1 một mình chỉ dừng ở mức 5%, nên chọn d=1+D=1 an toàn hơn, và std giảm đều (26.8 → 15.8 → 9.9) nên không có dấu hiệu sai phân thừa.
- **Bàn giao:** hàm `make_stationary(series, d, D, m)` trên train **gốc** (không log, không chuẩn hóa) tái tạo chính xác chuỗi; chuỗi được lưu ở `tables/stationary_dataset{1,2}_train.csv` (có index thời gian, mất 13 điểm đầu: D1 từ 1992-08, D2 từ 1950-02) cùng bảng ADF `tables/adf_results_02.csv`. Dùng chuỗi đã xử lý này cho **ACF/PACF**; khi fit ARIMA/SARIMA thì đưa **train gốc** vào mô hình với `order=(p,1,q)`, `seasonal_order=(P,1,Q,12)`, **không sai phân trước** để tránh sai phân hai lần.
- **Hạn chế:** quyết định chỉ dựa vào ADF trên train (không dùng test); ADF có công suất thấp khi có mùa vụ nên nên đối chiếu thêm bằng ACF/PACF và chẩn đoán phần dư.

**Trả lời câu hỏi theo đề (Bài 2–3 và phần nâng cao):**

- *H₀ của ADF là gì? p-value cho biết điều gì? Chuỗi gốc có dừng không?* H₀: có nghiệm đơn vị (không dừng); p-value như giải thích ở mục 1. Cả D1 (p = 0.999) lẫn D2 (p = 0.917) đều **không dừng**, khớp với yêu cầu "ADF chuỗi gốc → không dừng" của phần nâng cao.
- *Sau sai phân bậc 1, chuỗi đã dừng chưa? d trong ARIMA là bao nhiêu?* Dataset 1: sau d=1, ADF p = 0.428 → **chưa dừng** (còn mùa vụ); chuỗi chỉ dừng khi thêm sai phân mùa vụ D=1 (p ≈ 0). Vì vậy **d = 1** cho thành phần thường (ARIMA/SARIMA), và D=1 cho phần mùa vụ của SARIMA; ARIMA thuần chỉ có d nên sẽ không loại được mùa vụ, đây là hạn chế cần lưu ý ở phần ARIMA.
- *Sai phân thường + mùa vụ (Dataset 2, gợi ý d=1, D=1, s=12):* được dữ liệu ủng hộ, ADF p ≈ 0 và vững với mọi cách chọn lag đã thử.